In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import mlflow
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import roc_auc_score, brier_score_loss
from src.split import load_features, get_train_test, get_cv_folds

SEED = 42
mlflow.set_tracking_uri("http://10.0.100.20:5000/")
mlflow.set_experiment("tutoring-paradox-r7")

2026/11/11 15:21:42 INFO mlflow.tracking.fluent: Experiment with name 'tutoring-paradox-r7' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/130', creation_time=1794432103508, effective_trace_archival_retention=None, experiment_id='130', last_update_time=1794432103508, lifecycle_stage='active', name='tutoring-paradox-r7', tags={}, trace_location=None, workspace='default'>

In [6]:
BASE = ["failed_t1", "bus_rider", "econ_disadvantaged", "english_learner", "iep",
        "subj_English Language Arts", "subj_Mathematics", "subj_Science"]
GPA = ["prior_year_gpa", "gpa_missing"]
REF = ["referred_before_t3"]
ATT = ["attendance_rate_before_t3", "attendance_missing"]

MODELS = {
    "M0_base":            (BASE,                   "Baseline: background + term 1 only"),
    "M1_referral":        (BASE + REF,             "Does referral alone help? (shows the paradox)"),
    "M2_gpa":             (BASE + GPA,             "Does prior GPA help?"),
    "M3_gpa_referral":    (BASE + GPA + REF,       "H1: does referral help once GPA is known?"),
    "M4_plus_attendance": (BASE + GPA + REF + ATT, "H2: does attendance add anything?"),
}

In [7]:
df = load_features()
train, test = get_train_test(df)
folds = get_cv_folds(train)

print("Train rows:", len(train), "| students:", train["student_id"].nunique())
print("Test rows:", len(test), "(not used here; Shree scores it once in R8)")
print("Folds:", len(folds))
print("Train failure rate:", f"{train['failed_t3'].mean():.1%}")

Train rows: 95991 | students: 31997
Test rows: 24009 (not used here; Shree scores it once in R8)
Folds: 5
Train failure rate: 10.2%


In [9]:
def prepare(data):
    """Turn the raw table into numbers naive Bayes can use."""
    X = data.copy()

    # subject_area -> three 0/1 columns
    X = pd.get_dummies(X, columns=["subject_area"], prefix="subj", dtype=int)

    # Attendance: missing when no sessions were scheduled (mostly non-referred students)
    X["attendance_missing"] = X["attendance_rate_before_t3"].isna().astype(int)
    X["attendance_rate_before_t3"] = X["attendance_rate_before_t3"].fillna(0)

    # True/False -> 1/0
    for col in X.select_dtypes("bool").columns:
        X[col] = X[col].astype(int)
    return X

train_X = prepare(train)
print(train_X.filter(like="subj_").sum())
print("Missing GPA:", train_X["prior_year_gpa"].isna().sum(),
      "| missing attendance:", train_X["attendance_missing"].sum())

subj_English Language Arts    31997
subj_Mathematics              31997
subj_Science                  31997
dtype: int64
Missing GPA: 2961 | missing attendance: 85142


In [10]:
def evaluate(name, features, reason):
    """Train one model on 4 folds, score it on the 5th, repeat 5 times, then log to MLflow."""

    # Empty lists to collect one score per fold (5 of each at the end)
    aucs, briers, train_aucs = [], [], []

    # folds = 5 pairs of (rows to train on, rows to score on)
    # tr = row numbers for training, va = row numbers for validation
    for tr, va in folds:

        # Pick out this fold's rows, keeping only this model's columns
        # .copy() so we don't change train_X itself when we fill GPA below
        Xtr = train_X.iloc[tr][features].copy()   # training features
        Xva = train_X.iloc[va][features].copy()   # validation features
        ytr = train_X.iloc[tr]["failed_t3"]        # training answers (1 = failed)
        yva = train_X.iloc[va]["failed_t3"]        # validation answers

        # Only models with GPA (M2, M3, M4) need this
        if "prior_year_gpa" in features:
            # Median GPA from the TRAINING rows only,
            # so the validation students don't influence the fill value
            med = Xtr["prior_year_gpa"].median()
            # Fill blanks in both sets with that same training median
            Xtr["prior_year_gpa"] = Xtr["prior_year_gpa"].fillna(med)
            Xva["prior_year_gpa"] = Xva["prior_year_gpa"].fillna(med)

        # Train naive Bayes on the training rows
        model = GaussianNB().fit(Xtr, ytr)

        # Predicted chance of failing (column 1 = probability of "failed")
        p_tr = model.predict_proba(Xtr)[:, 1]   # on data it trained on
        p_va = model.predict_proba(Xva)[:, 1]   # on data it has NOT seen

        # Score this fold
        train_aucs.append(roc_auc_score(ytr, p_tr))   # AUC on training data
        aucs.append(roc_auc_score(yva, p_va))         # AUC on unseen data (the real score)
        briers.append(brier_score_loss(yva, p_va))    # how close probabilities are (lower = better)

    # After all 5 folds: average the scores
    result = {
        "model": name,
        "cv_auc_mean": np.mean(aucs),            # main score
        "cv_auc_sd": np.std(aucs),               # spread across folds (the ± part)
        "brier_mean": np.mean(briers),
        "train_auc_mean": np.mean(train_aucs),
        "gap": np.mean(train_aucs) - np.mean(aucs),  # big gap = overfitting
    }

    # Save this model as one run in MLflow
    with mlflow.start_run(run_name=name):
        # Settings: what model, which columns, and why this model exists
        mlflow.log_params({"model_type": "GaussianNB",
                           "features": ",".join(features),
                           "n_features": len(features),
                           "reason": reason,
                           "n_folds": len(folds),
                           "split_seed": SEED})
        # Scores (everything in result except the name)
        mlflow.log_metrics({k: v for k, v in result.items() if k != "model"})
        # A small file with the score from each fold (the "files" part of the rubric)
        fold_table = pd.DataFrame({"fold": range(1, len(folds) + 1),
                                   "cv_auc": aucs, "brier": briers,
                                   "train_auc": train_aucs})
        mlflow.log_text(fold_table.to_csv(index=False), "fold_scores.csv")

    return result


# Run evaluate() once for each model in MODELS (M0 to M4)
results = [evaluate(name, feats, reason) for name, (feats, reason) in MODELS.items()]

# Put the 5 results in one table, one row per model
summary = pd.DataFrame(results).set_index("model")
print(summary.round(4).to_string())

🏃 View run M0_base at: http://10.0.100.20:5000/#/experiments/130/runs/c8a8e1bafddf4c96a22b8ed826dc99eb
🧪 View experiment at: http://10.0.100.20:5000/#/experiments/130
🏃 View run M1_referral at: http://10.0.100.20:5000/#/experiments/130/runs/da6065b502684bf6997e3d6942f8259f
🧪 View experiment at: http://10.0.100.20:5000/#/experiments/130
🏃 View run M2_gpa at: http://10.0.100.20:5000/#/experiments/130/runs/30ed386c2dcb48578adf91de67b3c82d
🧪 View experiment at: http://10.0.100.20:5000/#/experiments/130
🏃 View run M3_gpa_referral at: http://10.0.100.20:5000/#/experiments/130/runs/0726e51807524896959afdf73e4e024a
🧪 View experiment at: http://10.0.100.20:5000/#/experiments/130
🏃 View run M4_plus_attendance at: http://10.0.100.20:5000/#/experiments/130/runs/5d15bf15783743ed9a41b31429c02635
🧪 View experiment at: http://10.0.100.20:5000/#/experiments/130
                    cv_auc_mean  cv_auc_sd  brier_mean  train_auc_mean     gap
model                                                           

In [12]:
# Extra run: M4 without the attendance_missing flag
m5 = evaluate("M5_attendance_no_flag",
              BASE + GPA + REF + ["attendance_rate_before_t3"],
              "Is the attendance flag causing the M4 drop?")
summary.loc["M5_attendance_no_flag"] = {k: v for k, v in m5.items() if k != "model"}
print(summary.round(4).to_string())

🏃 View run M5_attendance_no_flag at: http://10.0.100.20:5000/#/experiments/130/runs/307d3b445f2645599034473a8235cf88
🧪 View experiment at: http://10.0.100.20:5000/#/experiments/130
                       cv_auc_mean  cv_auc_sd  brier_mean  train_auc_mean     gap
model                                                                            
M0_base                     0.7855     0.0038      0.1187          0.7855  0.0000
M1_referral                 0.7879     0.0043      0.1195          0.7879  0.0000
M2_gpa                      0.8735     0.0014      0.1104          0.8740  0.0005
M3_gpa_referral             0.8648     0.0017      0.1163          0.8653  0.0005
M4_plus_attendance          0.8509     0.0019      0.1507          0.8514  0.0005
M5_attendance_no_flag       0.8595     0.0019      0.1300          0.8600  0.0005


In [13]:
from pathlib import Path

# Get every run from this MLflow experiment as a table
runs = mlflow.search_runs(experiment_names=["tutoring-paradox-r7"])

# Keep run name, run id, parameters and metrics (what the rubric asks for)
cols = ["tags.mlflow.runName", "run_id"] + [c for c in runs.columns
        if c.startswith(("params.", "metrics."))]

# Save to results/mlflow_runs.csv (makes the folder if it doesn't exist)
Path("../results").mkdir(exist_ok=True)
runs[cols].to_csv("../results/mlflow_runs.csv", index=False)
print(len(runs), "runs exported")

6 runs exported
